# <span style="color:red"> Analyzing Pit Stop Efficiency and Finishing Position in F1 (2011 Australian Grand Prix) </span>

Group Members: Kayla Jackson, Miriam Brem, Alyssa Chai

<span style="color: blue; font-size: 1.5em;">Introduction:</span>

<font size = "4">

This project examines how pit stop performance influences race outcomes in Formula One, focusing specifically on the 2011 Australian Grand Prix. Pit stops are brief but strategically critical moments in a race, and small differences in timing or execution can significantly affect a driver’s position on the track. Our research question asks: **How does pit stop performance measured by the number of pit stops and the average pit stop duration affect a driver’s final finishing position?** This question is relevant because pit stop strategy is one of the few controllable performance factors shared across all teams, making it a rich context for quantitative analysis. The goal of this project is to explore these relationships using real Formula One race data and present clear, data-driven insights into how pit stop efficiency shapes competitive outcomes.



**"Insert high-level description of the results and the coming structure of the project here"**

Group Members: Kayla Jackson, Miriam Brem, Alyssa Chai

<span style="color: blue; font-size: 1.5em;">Data Description:</span>

<font size = "4">

To investigate this question, we use two tables from the Formula One dataset: pit_stops.csv and results.csv. Each row in pit_stops.csv represents an individual pit stop made by a driver during a race, containing variables such as the race ID, driver ID, stop number, lap number, and pit stop duration in both string and millisecond formats. Each row in results.csv corresponds to a driver’s final race classification and includes information such as race ID, driver ID, starting grid position, finishing position, points earned, race completion status, and fastest lap time. The pit_stops.csv file contains all pit stops recorded across multiple seasons, while results.csv includes final results for every driver in every race. After importing both datasets into Python, we examine their dimensions, inspect the first rows, and prepare them for later merging using the shared keys raceId and driverId.





In [1]:
# import necessary libraries
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [5]:
# Load datasets
pit_stops = pd.read_csv("F1Data/pit_stops.csv")
results = pd.read_csv("F1Data/results.csv")

pit_stops.shape, results.shape

((9299, 7), (25660, 18))

The pit_stops table contains 9,299 observations and 7 variables, where each row represents a single pit stop made by a driver during a race. The results table contains 25,660 observations and 18 variables, with each row corresponding to a driver’s final race classification in a particular Grand Prix. Before merging the tables, we first examine both datasets for missing values, inconsistent formats, and variables that will not be used in our analysis.

In [ ]:
#inspec first rows of pit stops
pit_stops.head()


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [8]:
#inspect first rows of results
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


<span style="color: blue; font-size: 1.5em;">Data Cleaning:</span>

<font size = "4">
Before merging the tables, we carry out basic data cleaning to ensure that the variables used in the analysis are consistent and usable. This includes removing columns that will not be used, converting variables stored as strings into numeric formats, and handling missing values such as the placeholder “\N” that appears in the raw dataset. Since we plan to analyze pit stop timing and finishing position, we retain only the variables relevant to these topics and drop the remaining fields.

In [11]:
#Clean Code for pit_stops
# Keep only columns needed later 
pit_stops_clean = pit_stops[[
    "raceId",
    "driverId",
    "stop",
    "milliseconds"
]].copy()

#Replace "\N" with real NaN
pit_stops_clean = pit_stops_clean.replace("\\N", np.nan)

# Convert numeric columns
pit_stops_clean["raceId"] = pd.to_numeric(pit_stops_clean["raceId"], errors="coerce")
pit_stops_clean["driverId"] = pd.to_numeric(pit_stops_clean["driverId"], errors="coerce")
pit_stops_clean["stop"] = pd.to_numeric(pit_stops_clean["stop"], errors="coerce")
pit_stops_clean["milliseconds"] = pd.to_numeric(pit_stops_clean["milliseconds"], errors="coerce")



# Show summary
pit_stops_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9299 entries, 0 to 9298
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   raceId        9299 non-null   int64
 1   driverId      9299 non-null   int64
 2   stop          9299 non-null   int64
 3   milliseconds  9299 non-null   int64
dtypes: int64(4)
memory usage: 290.7 KB


In [10]:
# Clean code for results

# Keep only variables relevant later
results_clean = results[[
    "resultId",
    "raceId",
    "driverId",
    "position",
    "points",
    "milliseconds"
]].copy()

# Replace "\N" with NaN
results_clean = results_clean.replace("\\N", np.nan)

# Convert numeric columns
numeric_cols = ["resultId",
    "raceId",
    "driverId",
    "position",
    "points",
    "milliseconds"]

for col in numeric_cols:
    results_clean[col] = pd.to_numeric(results_clean[col], errors="coerce")

# Summary
results_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25660 entries, 0 to 25659
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   resultId      25660 non-null  int64  
 1   raceId        25660 non-null  int64  
 2   driverId      25660 non-null  int64  
 3   position      14833 non-null  float64
 4   points        25660 non-null  float64
 5   milliseconds  6963 non-null   float64
dtypes: float64(3), int64(3)
memory usage: 1.2 MB


Merging Data

In order to connect pit stop performance to race outcomes, we need a single dataset that combines information from both `pit_stops_clean` and `results_clean`. We do this by merging the two tables on the shared keys `raceId` and `driverId`. This ensures that each row in the merged dataset corresponds to a specific driver in a specific race, along with all of their recorded pit stops and final race result.


In [15]:
# Merging Data

# Merge pit stop data with race result data
merged = pit_stops_clean.merge(
    results_clean,
    on=["raceId", "driverId"],
    how="inner"   # keep only driver–race pairs that appear in both tables
)

#keeps only data for 2011 Australian Grand Prix
race_data = merged.query("raceId == 841").copy()

# Check the shape of the merged dataset
race_data.shape

race_data.head()

,raceId,driverId,stop,milliseconds_x,resultId,position,points,milliseconds_y
0,841,153,1,26898,20789,11.0,0.0,NaN
1,841,30,1,25021,20797,NaN,0.0,NaN
2,841,17,1,23426,20783,5.0,10.0,5408430.0
3,841,4,1,23251,20782,4.0,12.0,5402031.0
4,841,13,1,23842,20785,7.0,6.0,5455445.0


Cleaning Summary

The two cleaned tables were merged using an inner join on raceId and driverId, so the merged dataset only includes driver-race combinations that appear in both the pit stop records and the final race results. As a result, each row in the merged dataset represents a single pit stop made by a specific driver in a specific race, along with that driver’s final finishing position, race time, and points. After merging, we filtered the data to keep only observations from the 2011 Australian Grand Prix (raceId = 841) in order to centralize our analysis around a single race and directly address our research question. This structure will allow us to examine how pit stop performance relates to race outcomes in later sections.

<span style="color: red; font-size: 1.5em;">Notes for next steps (delete):</span>

<font size = "4">

There are two miliseconds milliseconds_x is how long this pit stop took
milliseconds_y is from results how long the entire race took for this driver (I think??). 

Things to maybe do:

-Rename Columns for Clarity

-Create New Variables (avg stop time, total # of stops, optional: total pit stop time)

-Use Groupby to collect info by Driver

-Table of descriptive statistcs????

-pargraph describing main columns 

-can sort variables like fastest to slowest

-loops and functions (can make one to summarize drivers stats)